# Build star-stack panels (Universe D)

After H-005 + H-006 STARs are frozen, run once to cache panels with the full stack applied.
Writes `s2_panel_D_1d_{train,full}_star.parquet` and manifest JSON.

H-007+ notebooks should `load_star_panels()` when manifest matches `s2_star_stack.json`.

In [ ]:
from __future__ import annotations

import json
import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    filter_pairs,
    frozen_pairs_for_universe,
    load_star_stack,
    load_universe_panels,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    lookbacks_for_bar,
    star_panel_paths,
    write_star_panel_manifest,
)
from backtest.s2_coint.report import load_star_stack

stack = load_star_stack(DEFAULT_STAR_STACK)
universe = str(stack["UNIVERSE_STAR"])
bar = str(stack.get("BAR_STAR") or "1d")
pair_ids = frozen_pairs_for_universe(universe, bar=bar)
train, full = load_universe_panels(universe, bar, pair_ids)

lb = lookbacks_for_bar(bar)
if stack.get("HEDGE_STAR") == "kalman":
    train = overlay_kalman_hedge(
        train,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )
    full = overlay_kalman_hedge(
        full,
        burn_in=lb["kalman_burn_in"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )
else:
    train = overlay_ols_hedge(
        train,
        ols_window=lb["ols_window"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )
    full = overlay_ols_hedge(
        full,
        ols_window=lb["ols_window"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )

train_path, full_path, manifest_path = star_panel_paths(universe=universe, bar=bar)
os.makedirs(os.path.dirname(train_path), exist_ok=True)
train.to_parquet(train_path, index=False)
full.to_parquet(full_path, index=False)
write_star_panel_manifest(manifest_path, stack, pair_ids=pair_ids)
print(f"wrote {train_path}")
print(f"wrote {full_path}")
print(f"wrote {manifest_path}")